In [10]:
import copy
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d

from evo.tools import file_interface, plot
from evo.core import metrics
from evo.core.units import Unit
from evo.tools.settings import SETTINGS
from evo.core.metrics import PoseRelation
from matplotlib import cm,colors
from evo.core import geometry

1.traj

In [11]:
def _paper_axes(ax):

    ax.figure.set_facecolor("white")
    ax.set_facecolor("white")

    ax.set_axisbelow(True)
    ax.minorticks_on()
    ax.grid(True, which="major", color="0.85", linestyle="--", linewidth=0.7)
    ax.grid(True, which="minor", color="0.75", linestyle="--", linewidth=0.3)

    for s in ax.spines.values():
        s.set_color("black")
        s.set_linewidth(2.0)

    ax.tick_params(which="major", direction="in", length=7, width=2, colors="black", top=True, right=True)
    ax.tick_params(which="minor", direction="in", length=4, width=1.4, colors="black", top=True, right=True)


def _paper_legend(ax, ncol=1, loc="upper right", bbox=None, fontsize=14):
    kwargs = dict(
        loc=loc,
        ncol=ncol,
        fontsize=fontsize,
        frameon=True,
        fancybox=False,
        framealpha=1.0,
        edgecolor="black",
        facecolor="white",
        borderpad=0.6,
        handlelength=2.6,
        handletextpad=0.6,
        columnspacing=1.2,
    )
    if bbox is not None:
        kwargs["bbox_to_anchor"] = bbox
    leg = ax.legend(**kwargs)
    leg.get_frame().set_linewidth(1.8)
    return leg


def evo_traj_plot(gt_path, pred_path):

    traj_ref = file_interface.read_kitti_poses_file(gt_path)
    traj_est = file_interface.read_kitti_poses_file(pred_path)

    # alignment demo align()
    traj_est_aligned = copy.deepcopy(traj_est)
    traj_est_aligned.align(traj_ref)

    plot_collection = plot.PlotCollection("traj")

    # trajectory
    fig1 = plt.figure(figsize=(7, 7))

    plot_mode = plot.PlotMode.xy #xz / yz / xyz
    ax1 = plot.prepare_axis(fig1, plot_mode)

    plot.traj(ax1, plot_mode, traj_ref, "--", "gray", label="GroundTruth Poses")
    plot.traj(ax1, plot_mode, traj_est_aligned, "-", "firebrick", label="Pred Poses")

    ax1.set_title("Trajectory")

    _paper_axes(ax1)
    _paper_legend(ax1, ncol=1, loc="upper right", bbox=None, fontsize=20)

    plot_collection.add_figure("traj", fig1)


    # xyz
    fig_xyz, axarr_xyz = plt.subplots(3, sharex="col", figsize=(10, 6), dpi=100)

    SETTINGS.plot_linewidth = 1.0
    plot.apply_settings(SETTINGS)
    length_unit = Unit.meters

    plot.traj_xyz(
        axarr_xyz,
        traj_ref,
        style="--",
        color="gray",
        label="GroundTruth Poses",
        alpha=1.0,
        start_timestamp=None,
        length_unit=length_unit,
    )

    plot.traj_xyz(
        axarr_xyz,
        traj_est_aligned,  
        style="-",
        color="firebrick",
        label="Pred Poses",
        alpha=1.0,
        start_timestamp=None,
        length_unit=length_unit,
    )


    for ax in axarr_xyz:
        _paper_axes(ax)
    for ax in axarr_xyz:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    handles, labels = axarr_xyz[0].get_legend_handles_labels()
    for ax in axarr_xyz:
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    leg = axarr_xyz[0].legend(
        loc="upper right",
        fontsize=20,
        frameon=True,
        fancybox=False,
        framealpha=1.0,
        edgecolor="black",
        facecolor="white",
    )
    leg.get_frame().set_linewidth(1.0)



    axarr_xyz[0].set_title("XYZ components")
    fig_xyz.tight_layout()

    plot_collection.add_figure("xyz", fig_xyz)

    plot_collection.show()

In [ ]:
# GT = r"D:\project2\data\poses_data\round1\BA_result_with_deskewed_round1_4470.txt"
# EST = r"D:\project2\data\poses_data\round1\self_online_pose_without_deskew.txt"
# EST = r"D:\project2\data\poses_data\round1\round1_ekf.txt"
# evo_traj_plot(GT, EST)


#round2
GT = r"D:\project2\data\poses_data\round2\BA_result_with_deskewed_round2_4259.txt"
# EST = r"D:\project2\data\poses_data\round2\pred_round2_without_deskewed.txt"
#prebuild map
EST = r"D:\project2\evaluation\data\round2\sequence02_estimated.txt"
evo_traj_plot(GT, EST)

C:\Users\75264\AppData\Local\Temp\ipykernel_40172\803526849.py:124: UserWarning: The figure layout has changed to tight
  fig_xyz.tight_layout()


2. ape

In [13]:
def evo_ape_plot(gt_path, est_path):

    traj_ref = file_interface.read_kitti_poses_file(gt_path)
    traj_est = file_interface.read_kitti_poses_file(est_path)

    traj_est_aligned = copy.deepcopy(traj_est)
    traj_est_aligned.align(traj_ref)

    ape = metrics.APE(pose_relation=metrics.PoseRelation.translation_part)
    ape.process_data((traj_ref, traj_est_aligned))
    ape_stats = ape.get_all_statistics()

    ape_stats.pop("sse", None)
    ape_stats.pop("mean", None)
    ape_stats.pop("median", None)
    ape_stats.pop("std", None)

    print("rmse:", ape_stats["rmse"])
    print("min:", ape_stats["min"])
    print("max:", ape_stats["max"])

    # ===========fig===========

    plot_collection = plot.PlotCollection("APE Resutls")

    #==========fig_error==========
    fig1 = plt.figure(figsize=(8,8))
    ax = fig1.gca()

    plot.error_array(
        ax,
        ape.error,
        statistics=ape_stats,
        name="APE",
        title="APE Evaluation Result"
    )

    _paper_axes(ax)
    _paper_legend(ax, ncol=1, loc="lower right", bbox=None, fontsize=20)

    plot_collection.add_figure("error", fig1)

    #============traj&ape===============
    fig2 = plt.figure(figsize=(8,8))
    plot_mode = plot.PlotMode.xyz    #can change to xyz,xy...
    ax = plot.prepare_axis(fig2, plot_mode)
    plot.traj(ax,plot_mode, traj_ref, "--", "gray", "GroundTruth Trajectory")

    plot.traj_colormap(
        ax,
        traj_est_aligned,
        ape.error,
        plot_mode,
        min_map=ape_stats["min"],
        max_map=ape_stats["max"],
        title="APE mapped onto trajectory",
    )
    _paper_axes(ax)
    _paper_legend(ax, ncol=1, loc="upper right", bbox=None, fontsize=20)

    plot_collection.add_figure("traj (error)", fig2)

    plot_collection.show()


In [19]:
GT = r"D:\project2\data\poses_data\round1\BA_result_with_deskewed_round1_4470.txt"
#EST = r"D:\project2\data\poses_data\round1\self_online_pose_without_deskew.txt"
EST = r"D:\project2\data\poses_data\round1\round1_ekf.txt"

#round2
# GT = r"D:\project2\data\poses_data\round2\BA_result_with_deskewed_round2_4259.txt"
# EST = r"D:\project2\data\poses_data\round2\pred_round2_without_deskewed.txt"

evo_ape_plot(GT, EST)

rmse: 0.9366645602910544
min: 0.10882882662133014
max: 1.767379631815789


3. RPE

In [15]:
def evo_rpe_plot(gt_path, est_path):

    traj_ref = file_interface.read_kitti_poses_file(gt_path)
    traj_est = file_interface.read_kitti_poses_file(est_path)

    traj_est_aligned = copy.deepcopy(traj_est)
    traj_est_aligned.align(traj_ref)

    rpe = metrics.RPE(pose_relation=metrics.PoseRelation.translation_part)
    rpe.process_data((traj_ref, traj_est_aligned))
    rpe_stats = rpe.get_all_statistics()

    rpe_stats.pop("sse", None)
    rpe_stats.pop("mean", None)
    rpe_stats.pop("median", None)
    rpe_stats.pop("std", None)

    print("rmse:", rpe_stats["rmse"])
    print("min:", rpe_stats["min"])
    print("max:", rpe_stats["max"])

    # ===========fig===========

    plot_collection = plot.PlotCollection("RPE Resutls")

    #==========fig_error==========
    fig1 = plt.figure(figsize=(8,8))
    ax = fig1.gca()

    plot.error_array(
        ax,
        rpe.error,
        statistics=rpe_stats,
        name="RPE",
        title="RPE Evaluation Result"
    )

    _paper_axes(ax)
    _paper_legend(ax, ncol=1, loc="lower right", bbox=None, fontsize=20)

    plot_collection.add_figure("error", fig1)

    #============traj&ape===============
    fig2 = plt.figure(figsize=(8,8))
    plot_mode = plot.PlotMode.xyz    #can change to xyz,xy...
    ax = plot.prepare_axis(fig2, plot_mode)
    plot.traj(ax,plot_mode, traj_ref, "--", "gray", "GroundTruth Trajectory")

    plot.traj_colormap(
        ax,
        traj_est_aligned,
        rpe.error,
        plot_mode,
        min_map=rpe_stats["min"],
        max_map=rpe_stats["max"],
        title="RPE mapped onto trajectory",
    )
    _paper_axes(ax)
    _paper_legend(ax, ncol=1, loc="upper right", bbox=None, fontsize=20)

    plot_collection.add_figure("traj (error)", fig2)

    plot_collection.show()


In [16]:
GT = r"D:\project2\data\poses_data\round1\BA_result_with_deskewed_round1_4470.txt"
# EST = r"D:\project2\data\poses_data\round1\self_online_pose_without_deskew.txt"
EST = r"D:\project2\data\poses_data\round1\round1_ekf.txt"

#round2
# GT = r"D:\project2\data\poses_data\round2\BA_result_with_deskewed_round2_4259.txt"
# EST = r"D:\project2\data\poses_data\round2\pred_round2_without_deskewed.txt"

evo_rpe_plot(GT, EST)

rmse: 0.06906420279281754
min: 0.0024999340820904244
max: 0.347140408201679


4. trajectory error in mesh result

In [17]:
def make_tube_from_polyline(points_xyz, seg_pairs, seg_rgb, radius=0.08, resolution=10):
    tube = o3d.geometry.TriangleMesh()
    pts = np.asarray(points_xyz)

    for (i, j), col in zip(seg_pairs, seg_rgb):
        p0 = pts[i]; p1 = pts[j]
        v = p1 - p0
        L = float(np.linalg.norm(v))
        if L <= 1e-9:
            continue

        cyl = o3d.geometry.TriangleMesh.create_cylinder(radius=radius, height=L, resolution=resolution)
        cyl.paint_uniform_color(col.tolist())

        z = np.array([0.0, 0.0, 1.0], dtype=float)
        d = v / L
        axis = np.cross(z, d)
        axis_norm = np.linalg.norm(axis)
        if axis_norm > 1e-12:
            axis = axis / axis_norm
            angle = float(np.arccos(np.clip(np.dot(z, d), -1.0, 1.0)))
            R = o3d.geometry.get_rotation_matrix_from_axis_angle(axis * angle)
            cyl.rotate(R, center=np.zeros(3))
        elif np.dot(z, d) < 0:
            R = o3d.geometry.get_rotation_matrix_from_axis_angle(np.array([1.0, 0.0, 0.0]) * np.pi)
            cyl.rotate(R, center=np.zeros(3))

        cyl.translate((p0 + p1) * 0.5)

        tube += cyl

    tube.compute_vertex_normals()
    return tube



def lift_traj_constant(pos_xyz: np.ndarray, lift_m: float) -> np.ndarray:
    pos2 = pos_xyz.copy()
    pos2[:, 2] += float(lift_m)
    return pos2

In [18]:
MESH_PATH = r"D:\project2\data\round1\mesh_result\gt.ply"     # .ply/.obj/.stl/.pcd 等
# GT_KITTI  = r"D:\project2\data\poses_data\round1\BA_result_with_deskewed_round1_4470.txt"     # KITTI poses: 每行 12 个数 (3x4)
# EST_KITTI = r"D:\project2\data\poses_data\round1\self_online_pose_without_deskew.txt"

#round2
GT_KITTI  = r"D:\project2\data\poses_data\round2\BA_result_with_deskewed_round2_4259.txt"    # KITTI poses: 每行 12 个数 (3x4)
EST_KITTI = r"D:\project2\data\poses_data\round2\pred_round2_without_deskewed.txt"


WITH_SCALE = False     # 单目/尺度未知时改 True（Sim(3)）
USE_RPE = False
RPE_DELTA = 1          # frames

traj_ref = file_interface.read_kitti_poses_file(GT_KITTI)
traj_est = file_interface.read_kitti_poses_file(EST_KITTI)

n = min(traj_ref.num_poses, traj_est.num_poses)
ids = np.arange(n, dtype=int)
traj_ref = copy.deepcopy(traj_ref); traj_ref.reduce_to_ids(ids)
traj_est = copy.deepcopy(traj_est); traj_est.reduce_to_ids(ids)

traj_est_eval = copy.deepcopy(traj_est)
traj_est_eval.align(traj_ref, correct_scale=WITH_SCALE, correct_only_scale=True)

if not USE_RPE:
    ape = metrics.APE(pose_relation=PoseRelation.translation_part)
    ape.process_data((traj_ref, traj_est_eval))
    err_pose = ape.error                    
    err_seg  = 0.5 * (err_pose[:-1] + err_pose[1:])  
else:
    rpe = metrics.RPE(
        pose_relation=PoseRelation.translation_part,
        delta=RPE_DELTA,
        delta_unit=Unit.frames,
        rel_delta_tol=0.1,
        all_pairs=False,
    )
    rpe.process_data((traj_ref, traj_est_eval))
    err_seg = rpe.error

pos_draw = traj_est.positions_xyz   

norm = colors.Normalize(vmin=float(np.min(err_seg)), vmax=float(np.max(err_seg)))
cmap = cm.get_cmap("turbo")
seg_rgb = cmap(norm(err_seg))[:, :3].astype(np.float64)

if not USE_RPE:
    seg_pairs = np.stack([np.arange(n-1), np.arange(1, n)], axis=1).astype(np.int32)
else:
    m = len(err_seg)
    i0 = np.arange(m, dtype=int)
    i1 = i0 + RPE_DELTA
    seg_pairs = np.stack([i0, i1], axis=1).astype(np.int32)


x = traj_est.positions_xyz.T   # shape (3, N)
y = traj_ref.positions_xyz.T
R, t, c = geometry.umeyama_alignment(x, y, with_scale=WITH_SCALE)

mesh = o3d.io.read_triangle_mesh(MESH_PATH)
mesh.compute_vertex_normals()

mesh.scale(c, center=(0,0,0))
mesh.rotate(R, center=(0,0,0))
mesh.translate(t.reshape(3))

pos_draw = traj_est_eval.positions_xyz
pos_draw = lift_traj_constant(pos_draw, lift_m=8.0)

tube = make_tube_from_polyline(pos_draw, seg_pairs, seg_rgb, radius=0.40, resolution=20)
mesh.paint_uniform_color([0.65, 0.65, 0.65])

o3d.visualization.draw_geometries([mesh, tube])

o3d.visualization.draw_geometries([mesh])


C:\Users\75264\AppData\Local\Temp\ipykernel_40172\603307932.py:44: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap("turbo")
